## Import Libraries

In [5]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler
)

from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    ConfusionMatrixDisplay
)
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

## Load Feature Engineered Dataset

In [6]:
DATA_PATH = r"D:\Telco_Customer_Churn\data\processed_data\churn_feature_engineered.csv"
FIGURES_DIR = r"D:\Telco_Customer_Churn\reports\figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(3)

Dataset loaded: 7043 rows x 25 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TotalServicesSubscribed,CustomerTenureGroup,CustomerValueTier,DigitalAdoptionScore
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,2,New Customer,Low Value,2
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,No,4,Regular Customer,Medium Value,1
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,4,New Customer,Low Value,3


# Dataset Overview

Understand the dataset before model training.

We will inspect:

- Dataset Shape
- Sample Records
- Data Types
- Missing Values

In [7]:
# Shape
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

# First five rows
df.head()

Rows    : 7043
Columns : 25


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TotalServicesSubscribed,CustomerTenureGroup,CustomerValueTier,DigitalAdoptionScore
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,2,New Customer,Low Value,2
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,No,4,Regular Customer,Medium Value,1
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,4,New Customer,Low Value,3
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,4,Regular Customer,Medium Value,1
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,2,New Customer,Low Value,1


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerID               7043 non-null   str    
 1   gender                   7043 non-null   str    
 2   SeniorCitizen            7043 non-null   int64  
 3   Partner                  7043 non-null   str    
 4   Dependents               7043 non-null   str    
 5   tenure                   7043 non-null   int64  
 6   PhoneService             7043 non-null   str    
 7   MultipleLines            7043 non-null   str    
 8   InternetService          7043 non-null   str    
 9   OnlineSecurity           7043 non-null   str    
 10  OnlineBackup             7043 non-null   str    
 11  DeviceProtection         7043 non-null   str    
 12  TechSupport              7043 non-null   str    
 13  StreamingTV              7043 non-null   str    
 14  StreamingMovies          7043 non-n

In [9]:
1
df.isnull().sum()

customerID                 0
gender                     0
SeniorCitizen              0
Partner                    0
Dependents                 0
tenure                     0
PhoneService               0
MultipleLines              0
InternetService            0
OnlineSecurity             0
OnlineBackup               0
DeviceProtection           0
TechSupport                0
StreamingTV                0
StreamingMovies            0
Contract                   0
PaperlessBilling           0
PaymentMethod              0
MonthlyCharges             0
TotalCharges               0
Churn                      0
TotalServicesSubscribed    0
CustomerTenureGroup        0
CustomerValueTier          0
DigitalAdoptionScore       0
dtype: int64

## Review Engineered Features

In [10]:
engineered_features = [
    'TotalServicesSubscribed',
    'CustomerTenureGroup',
    'CustomerValueTier',
    'DigitalAdoptionScore'
]

df[engineered_features].head()

,TotalServicesSubscribed,CustomerTenureGroup,CustomerValueTier,DigitalAdoptionScore
0,2,New Customer,Low Value,2
1,4,Regular Customer,Medium Value,1
2,4,New Customer,Low Value,3
3,4,Regular Customer,Medium Value,1
4,2,New Customer,Low Value,1


## Feature Selection

In [11]:
# Remove identifier column
df = df.drop(columns=['customerID'])

In [12]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TotalServicesSubscribed,CustomerTenureGroup,CustomerValueTier,DigitalAdoptionScore
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,2,New Customer,Low Value,2
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,One year,No,Mailed check,56.95,1889.50,No,4,Regular Customer,Medium Value,1
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,4,New Customer,Low Value,3
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,4,Regular Customer,Medium Value,1
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,2,New Customer,Low Value,1


## Separate Features and Target

In [13]:
# Features
X = df.drop(columns=['Churn'])

# Target
y = df['Churn']

In [14]:
print("Features Shape :", X.shape)
print("Target Shape   :", y.shape)

Features Shape : (7043, 23)
Target Shape   : (7043,)


In [15]:
print(X.columns.tolist())

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'TotalServicesSubscribed', 'CustomerTenureGroup', 'CustomerValueTier', 'DigitalAdoptionScore']


In [16]:
y.value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

## Train-Test Split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
                                                X,
                                                y,
                                                test_size=0.2,
                                                random_state=42,
                                                stratify=y
                                            )

In [18]:
print(f"X train    : {X_train.shape}")
print(f"X test : {X_test.shape}")
print(f"Y Train    : {y_train.shape}")
print(f"Y test : {y_test.shape}")

X train    : (5634, 23)
X test : (1409, 23)
Y Train    : (5634,)
Y test : (1409,)


In [19]:
print(f"Y original  :  {y.value_counts()}")
print(f"Y train  :  {y_train.value_counts()}")
print(f"Y test  :  {y_test.value_counts()}")

Y original  :  Churn
No     5174
Yes    1869
Name: count, dtype: int64
Y train  :  Churn
No     4139
Yes    1495
Name: count, dtype: int64
Y test  :  Churn
No     1035
Yes     374
Name: count, dtype: int64


## Data Preprocessing

### Encode Target Variable


The target variable (`Churn`) contains categorical values:

- Yes
- No

Most machine learning algorithms in scikit-learn require the target variable to be numerical. Therefore, we encode:

- No → 0
- Yes → 1

This encoding is applied after the train-test split to maintain a clean and reproducible preprocessing workflow.

In [20]:
# Initialize Label Encoder
label_encoder = LabelEncoder()

In [21]:
# Encode the target variable

y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

In [22]:
# Verify encoded classes

print("Classes:", label_encoder.classes_)

Classes: ['No' 'Yes']


In [23]:
# Verify encoded target distribution

print("Training Target Distribution")
print(pd.Series(y_train).value_counts())

print("\nTesting Target Distribution")
print(pd.Series(y_test).value_counts())

Training Target Distribution
0    4139
1    1495
Name: count, dtype: int64

Testing Target Distribution
0    1035
1     374
Name: count, dtype: int64


### Identify Feature Types

Before preprocessing, we classify features based on their data type and business meaning.

This helps us apply the appropriate preprocessing technique to each group.

- **Numerical Features:** Continuous or count-based variables.
- **Binary Categorical Features:** Two possible categories (Yes/No, Male/Female).
- **Nominal Categorical Features:** Multiple categories with no natural order.
- **Ordinal Categorical Features:** Categories with a meaningful order.

In [24]:
# Numerical Features

numerical_features = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
    'TotalServicesSubscribed',
    'DigitalAdoptionScore'
]

print("Number of Numerical Features :", len(numerical_features))
print(numerical_features)

Number of Numerical Features : 5
['tenure', 'MonthlyCharges', 'TotalCharges', 'TotalServicesSubscribed', 'DigitalAdoptionScore']


In [25]:
# Binary Categorical Features

binary_features = [
    'gender',
    'SeniorCitizen',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling'
]

print("Number of Binary Features :", len(binary_features))
print(binary_features)

Number of Binary Features : 6
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']


In [26]:
# Nominal Categorical Features

nominal_features = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
]

print("Number of Nominal Features :", len(nominal_features))
print(nominal_features)

Number of Nominal Features : 10
['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']


In [27]:
# Ordinal Features

ordinal_features = [
    'CustomerTenureGroup',
    'CustomerValueTier'
]

print("Number of Ordinal Features :", len(ordinal_features))
print(ordinal_features)

Number of Ordinal Features : 2
['CustomerTenureGroup', 'CustomerValueTier']


In [28]:
total_features = (
    len(numerical_features)
    + len(binary_features)
    + len(nominal_features)
    + len(ordinal_features)
)

print("Total Features:", total_features)
print("Actual Features:", X_train.shape[1])

Total Features: 23
Actual Features: 23


## Build Preprocessing Pipelines

### Binary Feature Encoding

Gender
Partner
Dependents
PhoneService
PaperlessBilling
SeniorCitizen

In [29]:
# Binary Mapping

binary_mapping = {
    'Yes': 1,
    'No': 0,
    'Male': 1,
    'Female': 0
}

In [30]:
binary_string_features = [
    'gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling'
]

In [31]:
for col in binary_string_features:
    X_train[col] = X_train[col].map(binary_mapping)
    X_test[col] = X_test[col].map(binary_mapping)

In [32]:
X_train[binary_string_features].head()

,gender,Partner,Dependents,PhoneService,PaperlessBilling
3738,1,0,0,0,0
3151,1,1,1,1,0
4860,1,1,1,0,0
3867,0,1,0,1,1
3810,1,1,1,1,0


In [33]:
X_train[binary_string_features].nunique()

gender              2
Partner             2
Dependents          2
PhoneService        2
PaperlessBilling    2
dtype: int64

In [36]:
for col in binary_string_features:
    print(f"{col}: {X_train[col].unique()}")

gender: [1 0]
Partner: [0 1]
Dependents: [0 1]
PhoneService: [0 1]
PaperlessBilling: [0 1]


# Ordinal Encoding

Ordinal features have categories with a meaningful order.

Examples:

CustomerTenureGroup

New Customer → Regular Customer → Loyal Customer

CustomerValueTier

Low Value → Medium Value → High Value

Since these categories have a natural ranking, we preserve this order by assigning increasing numerical values.

In [37]:
# Customer Tenure Group Mapping

tenure_mapping = {
    'New Customer': 0,
    'Regular Customer': 1,
    'Loyal Customer': 2
}

In [38]:
# Customer Value Tier Mapping

value_mapping = {
    'Low Value': 0,
    'Medium Value': 1,
    'High Value': 2
}

In [39]:
# Apply Ordinal Encoding

X_train['CustomerTenureGroup'] = X_train['CustomerTenureGroup'].map(tenure_mapping)
X_test['CustomerTenureGroup'] = X_test['CustomerTenureGroup'].map(tenure_mapping)

X_train['CustomerValueTier'] = X_train['CustomerValueTier'].map(value_mapping)
X_test['CustomerValueTier'] = X_test['CustomerValueTier'].map(value_mapping)

In [40]:
# Check first few rows

X_train[['CustomerTenureGroup', 'CustomerValueTier']].head()

,CustomerTenureGroup,CustomerValueTier
3738,1,1
3151,1,1
4860,1,0
3867,1,1
3810,0,0


In [41]:
print("CustomerTenureGroup :", sorted(X_train['CustomerTenureGroup'].unique()))
print("CustomerValueTier   :", sorted(X_train['CustomerValueTier'].unique()))

CustomerTenureGroup : [np.int64(0), np.int64(1), np.int64(2)]
CustomerValueTier   : [np.int64(0), np.int64(1), np.int64(2)]


In [43]:
# Check for missing values after mapping

print(X_train[['CustomerTenureGroup', 'CustomerValueTier']].isnull().sum())

print(X_test[['CustomerTenureGroup', 'CustomerValueTier']].isnull().sum())

CustomerTenureGroup    0
CustomerValueTier      0
dtype: int64
CustomerTenureGroup    0
CustomerValueTier      0
dtype: int64


# One-Hot Encoding

Nominal categorical features do not have a natural order.

Examples:

- Internet Service
- Payment Method
- Contract
- Multiple Lines

Assigning numerical values such as:

Electronic Check = 0
Mailed Check = 1
Bank Transfer = 2

would incorrectly imply that one category is greater than another.

One-Hot Encoding creates a separate binary column for each category, allowing machine learning models to treat each category independently.

In [44]:
nominal_features = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
]

print(nominal_features)

['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']


In [45]:
# Initialize One-Hot Encoder

onehot_encoder = OneHotEncoder(
    drop='first',
    handle_unknown='ignore',
    sparse_output=False
)

In [46]:
# Learn categories from training data

X_train_encoded = onehot_encoder.fit_transform(
    X_train[nominal_features]
)

In [47]:
# Apply learned categories to test data

X_test_encoded = onehot_encoder.transform(
    X_test[nominal_features]
)

In [48]:
encoded_feature_names = onehot_encoder.get_feature_names_out(
    nominal_features
)

In [49]:
X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoded_feature_names,
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoded_feature_names,
    index=X_test.index
)

In [50]:
X_train = X_train.drop(columns=nominal_features)
X_test = X_test.drop(columns=nominal_features)

In [51]:
X_train = pd.concat(
    [X_train, X_train_encoded],
    axis=1
)

X_test = pd.concat(
    [X_test, X_test_encoded],
    axis=1
)

In [52]:
print("Train Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)

Train Shape : (5634, 34)
Test Shape  : (1409, 34)


In [53]:
print(X_train.isnull().sum().sum())
print(X_test.isnull().sum().sum())

0
0


In [54]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 5634 entries, 3738 to 5639
Data columns (total 34 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 5634 non-null   int64  
 1   SeniorCitizen                          5634 non-null   int64  
 2   Partner                                5634 non-null   int64  
 3   Dependents                             5634 non-null   int64  
 4   tenure                                 5634 non-null   int64  
 5   PhoneService                           5634 non-null   int64  
 6   PaperlessBilling                       5634 non-null   int64  
 7   MonthlyCharges                         5634 non-null   float64
 8   TotalCharges                           5634 non-null   float64
 9   TotalServicesSubscribed                5634 non-null   int64  
 10  CustomerTenureGroup                    5634 non-null   int64  
 11  CustomerValueTier

In [55]:
print(X_train.shape)
print(X_test.shape)
print(X_train.info())

(5634, 34)
(1409, 34)
<class 'pandas.DataFrame'>
Index: 5634 entries, 3738 to 5639
Data columns (total 34 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 5634 non-null   int64  
 1   SeniorCitizen                          5634 non-null   int64  
 2   Partner                                5634 non-null   int64  
 3   Dependents                             5634 non-null   int64  
 4   tenure                                 5634 non-null   int64  
 5   PhoneService                           5634 non-null   int64  
 6   PaperlessBilling                       5634 non-null   int64  
 7   MonthlyCharges                         5634 non-null   float64
 8   TotalCharges                           5634 non-null   float64
 9   TotalServicesSubscribed                5634 non-null   int64  
 10  CustomerTenureGroup                    5634 non-null   int64  


# Feature Scaling

Machine learning algorithms such as Logistic Regression are sensitive to the scale of numerical features.

For example:

- tenure ranges from 0 to 72
- TotalCharges ranges from 0 to 8000+

Without scaling, features with larger values can dominate the learning process.

Tree-based algorithms such as Decision Tree and Random Forest do not require feature scaling because they split data based on thresholds rather than distances.

Therefore, scaling will only be applied to the Logistic Regression pipeline.